# 1.1 以分布式 GMRES 为例的 CANN/HPC 应用调优

> 实验主入口：[分布式 GMRES 单变量调优实验手册](EXPERIMENT_GUIDE.md)。先用 `msprof` 定位，再只改变一个变量，并保持正确性条件一致。

本章把 GMRES 作为包含 Ascend C SpMV、Dot、Norm、AXPY、Scale 与 HCCL collective 的综合 Device 工作负载，训练 baseline、profiling、瓶颈定位和单变量优化能力。CSR 与 Krylov 向量在 solve 内常驻 Device；Host 仅处理小型 Hessenberg/Givens 控制。

## 前置要求

- 完成 OpenMP、MPI、ACL 和 HCCL 章节
- 理解正确性与端到端计时
- 了解 GMRES 迭代包含稀疏和向量操作

## 本章学习目标

- 读取阶段 profiling 和 collective 次数
- 建立正确性与性能 baseline
- 从计算、通信、内存和调度提出可验证优化

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v msprof
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 章节内容

- [01.01_chapter_intro](01.01_chapter_intro.ipynb)：综合调优目标
- [01.02_distributed_gmres_profiling](01.02_distributed_gmres_profiling.ipynb)：阶段与 profiling 字段
- [01.03_baseline_and_bottleneck_analysis](01.03_baseline_and_bottleneck_analysis.ipynb)：正确性基线和瓶颈
- [01.04_compute_and_communication_tuning](01.04_compute_and_communication_tuning.ipynb)：MGS 与通信优化（CGS 运行时能力探测）
- [01.05_memory_scheduling_and_scaling](01.05_memory_scheduling_and_scaling.ipynb)：内存、同步、分区和 scaling
- [01.06_chapter_test](01.06_chapter_test.ipynb)：单变量优化闭环

## 实验材料说明

本章完整工程位于当前章节的 `src/`，练习参考位于 `answer/`。Notebook 使用相对路径访问材料，不依赖开发者本机的原始工程位置。

## 预期现象与结果分析

上面的目录检查应显示本章 Notebook、`answer`、`images` 和 `src`。若文件缺失，应先检查课程检出是否完整，而不是继续执行后续实验。按目录顺序学习，先确认正确性，再记录性能；不要用未启用真实后端的 stub 或 reference 路径代表 NPU 性能。

## 章节小结

本节给出了本章的能力目标、材料入口和学习顺序。下一节开始进入具体知识与实验。

## 实验工程说明与本章任务

本章实验工程位于 `src/dis_gmres/`。正式路径由 `npu_gmres.cpp`（Device-resident Arnoldi/GMRES）与 `npu_compute.cpp`（ACL RTC 编译、持久 DeviceVector 与 kernel launch）执行，SpMV/Dot/Norm/AXPY/Scale 来自 `kernels/gmres_ops.cpp` 的 Ascend C kernel；`gmres.cpp` 仅在无 CANN 时提供 Host stub 路径；`spmv.cpp` 只作正确性 reference；`hccl_comm.cpp` 是通信；`profiler.hpp` 计时；`main.cpp` 管理参数/正确性；脚本负责运行和 scaling。

### 本章实验任务

构建测试 → baseline → 核对 residual/iterations → 读 profiling → 找瓶颈 → 选择一个变量 → 只改该变量重跑 → 比较结论。

所有路径均相对 Notebook 当前目录。先检查环境，再运行真实工程；历史结果只用于观察趋势。

### 完成标准

能够指出核心源码和脚本职责，完成可用环境内的构建/诊断，并按“配置、Total、SpMV、Dot、AXPY、Norm、HCCL、Transfer、Residual、Iterations、Speedup”记录证据。
